# Análisis de la red de tiendas RetailNow

Análisis de ventas, inventario y satisfacción de clientes con **Pandas** y **NumPy**.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.float_format', lambda valor: f'{valor:,.2f}')

## Carga y limpieza de datos

Se usan las rutas absolutas requeridas por el enunciado. La alternativa local permite ejecutar el notebook desde la carpeta del proyecto.

In [ ]:
# El entorno del campus proporciona los CSV en estas rutas absolutas.
RUTA_BASE = Path('/workspace')

ventas = pd.read_csv(RUTA_BASE / 'sales.csv').dropna()
inventarios = pd.read_csv(RUTA_BASE / 'inventories.csv').dropna()
satisfaccion = pd.read_csv(RUTA_BASE / 'satisfaction.csv').dropna()

print(f'Ruta de datos: {RUTA_BASE}')
print(f'Filas válidas — ventas: {len(ventas)}, inventarios: {len(inventarios)}, satisfacción: {len(satisfaccion)}')
display(ventas.head())

## Análisis de ventas con Pandas

In [ ]:
# Total_Ventas representa los ingresos de cada operación.
ventas['Total_Ventas'] = ventas['Cantidad_Vendida'] * ventas['Precio_Unitario']
ventas_por_producto = ventas.groupby('Producto', as_index=False)['Total_Ventas'].sum()
ventas_por_tienda = ventas.groupby('ID_Tienda', as_index=False).agg(
    Unidades_Vendidas=('Cantidad_Vendida', 'sum'),
    Total_Ventas=('Total_Ventas', 'sum')
)
resumen_ventas = ventas.describe(include='all')

print('Ventas totales por producto')
display(ventas_por_producto)
print('Unidades vendidas e ingresos totales por tienda')
display(ventas_por_tienda)
print('Resumen estadístico de ventas')
display(resumen_ventas)

if 'Categoria' in ventas.columns:
    promedio_por_categoria = ventas.groupby(['ID_Tienda', 'Categoria'], as_index=False)['Total_Ventas'].mean()
    display(promedio_por_categoria)
else:
    print('El CSV no incluye la columna Categoria; este análisis condicional no aplica.')

## Rotación e inventario crítico

La rotación se calcula por tienda y producto como cantidad vendida entre stock disponible. Un valor inferior al 10 % se considera crítico según el enunciado.

In [ ]:
ventas_unidades = ventas.groupby(['ID_Tienda', 'Producto'], as_index=False).agg(
    Cantidad_Vendida=('Cantidad_Vendida', 'sum'),
    Total_Ventas=('Total_Ventas', 'sum')
)
inventarios = inventarios.merge(ventas_unidades, on=['ID_Tienda', 'Producto'], how='left')
inventarios[['Cantidad_Vendida', 'Total_Ventas']] = inventarios[['Cantidad_Vendida', 'Total_Ventas']].fillna(0)
inventarios['Rotacion_Inventario'] = np.where(
    inventarios['Stock_Disponible'] > 0,
    inventarios['Cantidad_Vendida'] / inventarios['Stock_Disponible'],
    np.nan
)
inventario_critico = inventarios[inventarios['Rotacion_Inventario'] < 0.10].copy()
rotacion_por_tienda = inventarios.groupby('ID_Tienda', as_index=False).agg(
    Unidades_Vendidas=('Cantidad_Vendida', 'sum'),
    Stock_Disponible=('Stock_Disponible', 'sum')
)
rotacion_por_tienda['Rotacion_Inventario'] = rotacion_por_tienda['Unidades_Vendidas'] / rotacion_por_tienda['Stock_Disponible']

print('Rotación por tienda y producto')
display(inventarios[['ID_Tienda', 'Producto', 'Stock_Disponible', 'Cantidad_Vendida', 'Rotacion_Inventario']])
print('Rotación agregada por tienda')
display(rotacion_por_tienda)
print('Registros con rotación crítica (< 10 %)')
display(inventario_critico if not inventario_critico.empty else pd.DataFrame({'Resultado': ['No se encontraron niveles críticos']}))

## Satisfacción del cliente y rendimiento

In [ ]:
rendimiento_tiendas = ventas_por_tienda.merge(satisfaccion, on='ID_Tienda', how='left')
tiendas_baja_satisfaccion = rendimiento_tiendas[rendimiento_tiendas['Satisfacción_Promedio'] < 60].copy()

display(rendimiento_tiendas)
if tiendas_baja_satisfaccion.empty:
    print('No hay tiendas con satisfacción inferior al 60 %.')
else:
    print('Tiendas con satisfacción baja:')
    display(tiendas_baja_satisfaccion)
    print('Recomendación: revisar atención al cliente, disponibilidad de productos y tiempos de espera; medir de nuevo tras aplicar mejoras.')

## Cálculos y simulación con NumPy

In [ ]:
array_ventas = ventas['Total_Ventas'].to_numpy()
mediana_ventas = np.median(array_ventas)
# np.std usa ddof=0 por defecto: desviación estándar poblacional.
desviacion_estandar_ventas = np.std(array_ventas)

# Semilla fija para obtener la misma simulación en cada ejecución.
generador = np.random.default_rng(seed=42)
proyecciones_futuras = generador.normal(
    loc=np.mean(array_ventas),
    scale=desviacion_estandar_ventas,
    size=12
)
proyecciones_futuras = np.clip(proyecciones_futuras, 0, None)
proyecciones_por_mes = pd.DataFrame({
    'Proyeccion_Ventas': proyecciones_futuras
}, index=pd.date_range('2024-01-01', periods=12, freq='MS'))
proyecciones_por_mes.index.name = 'Mes'

print(f'Mediana de las ventas totales: {mediana_ventas:,.2f} €')
print(f'Desviación estándar de las ventas totales: {desviacion_estandar_ventas:,.2f} €')
print('Proyecciones de ventas futuras (12 periodos):')
display(proyecciones_por_mes.round(2))
print(f'Media proyectada: {np.mean(proyecciones_futuras):,.2f} €')
print(f'Desviación estándar proyectada: {np.std(proyecciones_futuras):,.2f} €')

## Conclusiones

El notebook identifica los ingresos por tienda y producto, la rotación de stock y la relación entre ventas y satisfacción. Las tiendas con satisfacción inferior al 60 % deben priorizar acciones de servicio y disponibilidad. La simulación reproducible ofrece una referencia inicial para planificar ventas futuras.